# Intro to NN & PyTorch — Part 3: Training (Lessons 8–11)

> 📚 **Part of a 3-notebook set** (Codecademy *Intro to NN with PyTorch*): **Part 1 — Foundations** · **Part 2 — Building Networks** · **Part 3 — Training**. Each notebook is self-contained (run its Setup cell first).

The Loss Function (MSE) → Backward Pass & Optimizer Step → the Training Loop → Testing & Evaluation (train-test split, eval mode, save/load).

*Continues from Part 2 (Building Networks).*

In [1]:
# Setup — run once per session
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd

print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


torch version: 2.12.1+cu130
CUDA available: False


/home/plewis/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12050). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
# Recap from Part 2 — the NN_Regression class (used by the Lesson 9 demo below)
class NN_Regression(nn.Module):
    def __init__(self):
        super(NN_Regression, self).__init__()
        self.layer1 = nn.Linear(3, 16)
        self.layer2 = nn.Linear(16, 8)
        self.layer3 = nn.Linear(8, 4)
        self.layer4 = nn.Linear(4, 1)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.relu(self.layer3(x))
        x = self.layer4(x)
        return x


## Lesson 8 — The Loss Function

Our nets so far are **untrained**, so predictions are bad. To improve them — and to *track* improvement during training — we first need to measure **how bad** they are. That's the job of the **loss function**: a formula measuring the error (**loss**) between **predictions** and the **actual target values** (a.k.a. **labels**).

### Why not just take the difference?
For one apartment (actual \$1000, predicted \$500): difference = $500 - 1000 = -500$ (\$500 too low). Fine alone. But with a second (actual \$1500, predicted \$2000): difference = $2000 - 1500 = +500$.

Average those two → $\frac{-500 + 500}{2} = 0$ — falsely implying a **perfect** model. The problem: **negative and positive errors cancel.** We need to force every error **positive** first.

### Mean Squared Error (MSE)
The most common fix: **square** each difference (squares are always ≥ 0), then average.

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(\text{pred}_i - \text{target}_i)^2$$

For the two apartments:

$$\frac{(500-1000)^2 + (2000-1500)^2}{2} = \frac{250000 + 250000}{2} = 250{,}000$$

- The number is huge because errors were **squared**. To interpret it on the original scale, take the **square root** (RMSE): $\sqrt{250000} = 500$ → "off by \~\$500 on average," which matches intuition.

### MSE in PyTorch
PyTorch ships the common losses. Instantiate, then call with **(predictions, targets)**:

```python
loss = nn.MSELoss()
loss(predictions, y)
```

- **`y`** = the target values (ML convention, like `X` for inputs). **Lowercase `y`** here because targets are 1-D (a single value per example), vs. capital `X` for the 2-D input matrix.

> **⚠️ Gotcha — instantiate, *then* call (two steps, not one).** `nn.MSELoss(...)` is the **constructor**; it does **not** take your data. You build the loss object first, then call it with your tensors:
> ```python
> loss = nn.MSELoss()          # 1. build the loss object (no data here)
> MSE  = loss(predictions, y)  # 2. call it with (predictions, targets)
> # one-liner: nn.MSELoss()(predictions, y)  -> note the TWO sets of ()
> ```
> Passing tensors straight into the constructor — `nn.MSELoss(predictions, y)` — fails with:
> `RuntimeError: Boolean value of Tensor with more than one value is ambiguous`
> because PyTorch tries to read `predictions` as the constructor's `size_average` flag and evaluates a multi-element tensor as `True/False`. Same build-then-run pattern as a model: `model = NN_Regression()` **builds**, `model(X)` **runs**.

### Choosing a loss function
The loss steers training, so the choice matters — sometimes worth experimenting.
- **MSE** squaring **emphasizes the largest errors** (a big miss dominates the loss). Often helpful, but can drive **overfitting** to outliers.
- **MAE (Mean Absolute Error)** — uses **absolute value** instead of squaring (`nn.L1Loss`). Treats all errors proportionally → less sensitive to big outliers. For our example, MAE also = 500.

> Maps onto **3B1B Ch.2** — this is the **cost/loss function** whose value gradient descent will minimize. Next lesson starts the optimizer/backprop loop that actually drives this number down. See [[bb-nn-video-notes]].


In [3]:
# MSE loss on the two example apartments
loss = nn.MSELoss()

predictions = torch.tensor([500, 2000], dtype=torch.float)  # model's guesses
y          = torch.tensor([1000, 1500], dtype=torch.float)  # actual rents (targets)

mse = loss(predictions, y)
print("MSE :", mse.item())            # 250000.0  (errors squared -> large)
print("RMSE:", mse.sqrt().item())     # 500.0     (back on the $ scale)

# Same data with Mean Absolute Error (nn.L1Loss) for comparison
mae = nn.L1Loss()(predictions, y)
print("MAE :", mae.item())            # 500.0     (abs value instead of squaring)


MSE : 250000.0
RMSE: 500.0
MAE : 500.0


### Exercise — Computing loss (checkpoints 1–3)

1. **MSE by hand** — predictions `750, 1000` vs targets `1000, 900`. Square each difference, average → `36250.0`.
2. **MSE in PyTorch** — feed the *untrained* halving net's first 5 predictions vs the real rents through `nn.MSELoss()` → a giant `38,413,624` (untrained net's near-zero outputs are wildly off from \$2.5k–\$11.5k rents).
3. **RMSE** — square-root the MSE tensor (`MSE**(1/2)`) to read it on the dollar scale → `~6197.87` ≈ **off by ~\$6,200 per apartment**. Training will shrink this a lot.

> These are the exact `predicted_rent` values from the Lesson 7 class (`-6.9229, …`) paired with the real targets `y` — so this is literally "how wrong is our untrained model, in dollars."


In [4]:
# Checkpoint 1 — MSE by hand: predictions 750,1000 vs targets 1000,900
difference1 = 750 - 1000     # -250
difference2 = 1000 - 900     #  100

MSE = (difference1**2 + difference2**2) / 2   # square each, then average

MSE                          # 36250.0


36250.0

In [5]:
# Checkpoint 2 — MSE in PyTorch on the untrained net's first 5 predictions vs real rents
predictions = torch.tensor([-6.9229, -29.8163, -16.0748, -13.2427, -14.1096], dtype=torch.float)
y           = torch.tensor([2550, 11500, 3000, 4500, 4795], dtype=torch.float)

loss = nn.MSELoss()         # 1. build the loss object
MSE  = loss(predictions, y) # 2. call it with (predictions, targets)

print("MSE Loss:", MSE)     # tensor(38413624.) — huge: untrained net is way off


MSE Loss: tensor(38413624.)


In [6]:
# Checkpoint 3 — RMSE: square-root the MSE to get back to dollars
RMSE = MSE**(1/2)          # **(1/2) = square root of the tensor

RMSE                       # ~6197.87  -> model is off by ~$6,200 per apartment


tensor(6197.8726)

## Lesson 9 — Backward Pass & the Optimizer Step

We have predictions and a **loss** measuring how wrong they are. Training **lowers that loss** by repeatedly nudging the weights & biases "downhill." Each update has two moves:

- **Backward pass** — compute the **gradients** of the loss w.r.t. every weight/bias. The gradient points "uphill," so its negative is the **downward** direction (toward lower loss).
- **Step** — the **optimizer** uses those gradients to **update** the weights & biases a little in the downhill direction.

### Syntax
```python
MSE = loss(predictions, y)   # compute the loss
MSE.backward()               # backward pass -> fills in the gradients
optimizer.step()             # optimizer updates weights & biases using them
```

- **`.backward()` is called on the computed loss value, *not* the loss function.** That's only possible because the loss tensor carries a **`grad_fn`** — here **`grad_fn=<MseLossBackward0>`** — the function PyTorch uses to run the backward pass. (Every op records its `grad_fn`; chained together they form the graph autograd walks backward. This is what those `grad_fn=<...>` tags we kept seeing were for.)
- The **optimizer** (e.g. `optim.SGD(model.parameters(), lr=...)`) is set up *before* the loop — it's handed `model.parameters()` so `.step()` knows which tensors to update, and a **learning rate** controlling step size.

> ℹ️ **Note:** this fragment picked up mid-lesson; the **optimizer-definition** part is covered in the **next exercise below** (`optim.Adam(model.parameters(), lr=0.001)`). The demo cell here uses `optim.SGD` with a tiny `lr` as a simpler stand-in.
>
> ⚠️ Gradients **accumulate** by default, so a real loop also calls **`optimizer.zero_grad()`** before each `.backward()` (clears last step's gradients). Shown in the demo; likely the next topic.

> Maps onto **3B1B Ch.2–4**: `.backward()` = **backpropagation** computing the gradient; `optimizer.step()` = the **gradient-descent** update. See [[bb-nn-video-notes]].


In [7]:
# Demo: a few backward-pass + optimizer-step updates lowering the loss
import torch.optim as optim
import pandas as pd

torch.manual_seed(42)
model = NN_Regression()              # the 3->16->8->4->1 class from Lesson 7
loss = nn.MSELoss()

# faked StreeteEasy (same 5 apartments) + their real rents as targets
apartments_df = pd.DataFrame({
    "size_sqft":        [480, 2000, 1000, 916, 975],
    "bedrooms":         [0,   2,    3,    1,   1],
    "building_age_yrs": [17,  96,   106,  29,  31],
})
X = torch.tensor(apartments_df.values, dtype=torch.float32)
y = torch.tensor([2550, 11500, 3000, 4500, 4795], dtype=torch.float32).view(-1, 1)

# SCAFFOLDING (course defines the optimizer just before this fragment):
# tiny lr because the raw rent/sqft data is unnormalized -> big gradients.
optimizer = optim.SGD(model.parameters(), lr=1e-7)

# before any backward pass, gradients don't exist yet
print("grad before backward:", model.layer1.weight.grad)

for epoch in range(3):
    optimizer.zero_grad()            # clear last step's gradients (they accumulate)
    predictions = model(X)           # forward pass
    MSE = loss(predictions, y)       # compute the loss
    MSE.backward()                   # backward pass -> fills in .grad on every param
    optimizer.step()                 # update weights & biases downhill
    print(f"epoch {epoch}:  loss = {MSE.item():,.1f}   (grad_fn = {MSE.grad_fn})")

# after backward, gradients are populated
print("grad after backward (layer1, first row):", model.layer1.weight.grad[0])


grad before backward: None
epoch 0:  loss = 38,413,624.0   (grad_fn = <MseLossBackward0 object at 0x7e2afd8a8040>)
epoch 1:  loss = 38,196,736.0   (grad_fn = <MseLossBackward0 object at 0x7e2c100f1630>)
epoch 2:  loss = 38,196,724.0   (grad_fn = <MseLossBackward0 object at 0x7e2b0a2b3df0>)
grad after backward (layer1, first row): tensor([0., 0., 0.])


### Exercise — The optimizer (checkpoints 1–3)

*This supplies the optimizer setup that was missing from the pasted fragment above.*

1. **Import the optimizers:** `import torch.optim as optim` (already in our Setup cell).
2. **Initialize the optimizer:** `optimizer = optim.Adam(model.parameters(), lr=0.001)`.
   - **`model.parameters()`** — hands the optimizer every weight & bias to update.
   - **`lr=0.001`** — the **learning rate**: how big a step to take each update.
3. **One full update** on real apartment data: forward → loss → `MSE.backward()` → `optimizer.step()`, then re-run the forward pass to confirm the loss **dropped**.

**Notes:**
- **Adam** is a smarter optimizer than plain SGD: it **adapts the step size per-parameter** as it goes, so it handles awkwardly-scaled (unnormalized) data far better — which is why `lr=0.001` works cleanly here, whereas SGD needed `lr=1e-7` in the demo above to avoid diverging.
- **Learning rate trade-off:** too **small** → loss barely moves (slow); too **large** → can overshoot the minimum and diverge. Worth experimenting (the exercise invites trying other `lr` values).
- One step only nudges the loss a little — real training **loops** this many times.

> **⚠️ Faked data caveat:** CP3 runs on the **full** `streeteasy.csv` (all rows) with features `bedrooms, bathrooms, size_sqft`. Our 5-row fake stand-in **can't reproduce the course's exact numbers** (`29,213,900 → 29,205,546`) — those are over the whole dataset. We reproduce the **behavior** (Adam, `lr=0.001`, loss decreases after one step). Real course line kept as a comment.
>
> **⚠️ Shape note:** the course writes `y` as 1-D (`shape [N]`) while `predictions` is `[N,1]`, which makes `MSELoss` **broadcast** and emit a `UserWarning`. We use `y.view(-1, 1)` to match shapes cleanly (best practice).


In [8]:
# Checkpoint 1 — import PyTorch's optimizers as optim (already in Setup; here for completeness)
import torch.optim as optim


In [9]:
# Checkpoint 2 — initialize the Adam optimizer on the model's parameters
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(3, 16), nn.ReLU(),
    nn.Linear(16, 8), nn.ReLU(),
    nn.Linear(8, 4),  nn.ReLU(),
    nn.Linear(4, 1)
)

optimizer = optim.Adam(model.parameters(), lr=0.001)
optimizer


Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)

In [10]:
# Checkpoint 3 — one full optimization step: forward -> loss -> backward -> step
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(3, 16), nn.ReLU(),
    nn.Linear(16, 8), nn.ReLU(),
    nn.Linear(8, 4),  nn.ReLU(),
    nn.Linear(4, 1)
)

# import the data
# Real course code:
#   apartments_df = pd.read_csv("streeteasy.csv")
#   X = torch.tensor(apartments_df[['bedrooms','bathrooms','size_sqft']].values, dtype=torch.float)
#   y = torch.tensor(apartments_df['rent'].values, dtype=torch.float)
# FAKE 5-row stand-in (so numbers won't match the course's full-dataset values):
apartments_df = pd.DataFrame({
    "bedrooms":  [0, 2, 3, 1, 1],
    "bathrooms": [1, 2, 1, 1, 1],
    "size_sqft": [480, 2000, 1000, 916, 975],
    "rent":      [2550, 11500, 3000, 4500, 4795],
})
X = torch.tensor(apartments_df[['bedrooms', 'bathrooms', 'size_sqft']].values, dtype=torch.float)
y = torch.tensor(apartments_df['rent'].values, dtype=torch.float).view(-1, 1)  # view -> match [N,1]

# forward pass + initial loss
predictions = model(X)
loss = nn.MSELoss()
MSE = loss(predictions, y)
print('Initial loss is ' + str(MSE.item()))

# --- the optimization step ---
optimizer = optim.Adam(model.parameters(), lr=0.001)
MSE.backward()        # backward pass: gradients
optimizer.step()      # update weights & biases

# re-run forward pass with the updated weights -> new (lower) loss
predictions = model(X)
MSE = loss(predictions, y)
print('After optimization, loss is ' + str(MSE.item()))


Initial loss is 38282504.0
After optimization, loss is 38268116.0


## Lesson 10 — Training (the loop)

Last lesson did **one** optimization step. **Training** = repeating that step many times to drive the loss down as far as possible. Each pass through the loop is called an **epoch**.

### The four steps + one new line
```python
num_epochs = 1000
for epoch in range(num_epochs):
    predictions = model(X)        # 1. forward pass
    MSE = loss(predictions, y)    # 2. compute loss
    MSE.backward()                # 3. compute gradients (backward pass)
    optimizer.step()              # 4. update weights & biases
    optimizer.zero_grad()         # NEW: reset gradients for the next iteration
```

- **`optimizer.zero_grad()`** — the only new line. Gradients **accumulate** by default (PyTorch *adds* each `.backward()` onto the existing `.grad`). Without clearing them, each step's direction would be polluted by all previous steps. We want a **fresh direction each epoch**, so we zero them out. *(Order note: stepping then zeroing, or zeroing at the top of the loop, both work — just clear them once per iteration.)*
- **epoch** = one full iteration of the training loop (NN jargon).

### Tracking progress — print the loss periodically
```python
    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], MSE Loss: {MSE.item()}')
```
- **`epoch + 1`** because `epoch` starts at **0** -> the 100th epoch is `epoch == 99`. `(epoch + 1) % 100 == 0` fires on the 100th, 200th, ... epochs.
- **`.item()`** pulls out just the scalar loss value; printing `MSE` directly would also dump the `grad_fn`.

> **⚠️ Raw-data caveat (important):** on our **unnormalized** features (`size_sqft` in the hundreds/thousands), `lr=0.001` is tiny relative to that scale, so the loss **crawls** — it goes down, but barely, over 1000 epochs. The loop is *correct*; the **feature scaling** is the problem. This is exactly what the next lesson, **data normalization**, fixes. (Plus our 5-row fake is too small to learn much regardless.)

> Maps onto **3B1B**: this loop *is* gradient descent iterating — repeatedly stepping downhill on the cost surface. See [[bb-nn-video-notes]].


In [11]:
# Training loop — 1000 epochs of forward -> loss -> backward -> step -> zero_grad
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(3, 16), nn.ReLU(),
    nn.Linear(16, 8), nn.ReLU(),
    nn.Linear(8, 4),  nn.ReLU(),
    nn.Linear(4, 1)
)

# faked StreeteEasy stand-in (real course: pd.read_csv("streeteasy.csv"))
apartments_df = pd.DataFrame({
    "bedrooms":  [0, 2, 3, 1, 1],
    "bathrooms": [1, 2, 1, 1, 1],
    "size_sqft": [480, 2000, 1000, 916, 975],
    "rent":      [2550, 11500, 3000, 4500, 4795],
})
X = torch.tensor(apartments_df[['bedrooms', 'bathrooms', 'size_sqft']].values, dtype=torch.float)
y = torch.tensor(apartments_df['rent'].values, dtype=torch.float).view(-1, 1)

loss = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 1000
for epoch in range(num_epochs):
    predictions = model(X)          # forward pass
    MSE = loss(predictions, y)      # compute loss
    MSE.backward()                  # compute gradients
    optimizer.step()                # update weights & biases
    optimizer.zero_grad()           # reset gradients for the next iteration

    if (epoch + 1) % 100 == 0:      # print every 100 epochs
        print(f'Epoch [{epoch + 1}/{num_epochs}], MSE Loss: {MSE.item():.2f}')

# NOTE: loss barely moves -> unnormalized features make lr=0.001 too small.
# The training-loop code is correct; normalization (next lesson) is what unlocks real progress.


Epoch [100/1000], MSE Loss: 38195704.00
Epoch [200/1000], MSE Loss: 38194652.00


Epoch [300/1000], MSE Loss: 38193596.00
Epoch [400/1000], MSE Loss: 38192544.00


Epoch [500/1000], MSE Loss: 38191488.00
Epoch [600/1000], MSE Loss: 38190440.00


Epoch [700/1000], MSE Loss: 38189384.00
Epoch [800/1000], MSE Loss: 38188332.00


Epoch [900/1000], MSE Loss: 38187276.00
Epoch [1000/1000], MSE Loss: 38186220.00


### Exercise — Training loop debugging (checkpoints 1–3)

Now with **14 numerical features** (bedrooms, bathrooms, size_sqft, min_to_subway, floor, building_age_yrs, + 8 binary `has_*`/`no_fee` amenity flags) and a bigger net: **`14 → 128 → 64 → 1`** (still "halving": first hidden layer twice the second).

The two debugging checkpoints capture the **two classic training-loop bugs** — worth memorizing by their *symptom*:

| Symptom | Cause | Fix |
|---|---|---|
| **CP1: loss extremely high / barely improving** | **`optimizer.zero_grad()` missing** → gradients **accumulate** across epochs, so each step's direction is polluted by every prior step | add `optimizer.zero_grad()` once per iteration |
| **CP2: loss frozen (identical every epoch)** | the **update never happens** — either `MSE.backward()` is missing, or `zero_grad()`/`step()` is ordered so gradients are wiped **before** `step()` applies them | ensure order is `backward()` → `step()`, with `zero_grad()` **after** `step()` (or at the very top, before `backward()`) |

**The one rule that prevents both:** each epoch must run **`backward()` → `step()`** with a single **`zero_grad()`** that is *not* sandwiched between them. Correct body:

```python
predictions = model(X)        # forward
MSE = loss(predictions, y)    # loss
MSE.backward()                # gradients
optimizer.step()              # update
optimizer.zero_grad()         # clear for next epoch
```

3. **CP3** — just run the correct loop for **1000 epochs**. More epochs → lower loss, but with **diminishing returns** (the curve flattens; see the course's `3.13M → 2.63M`).

> **⚠️ Faked data caveat:** the course trains on the **full** `streeteasy.csv` (thousands of rows). I use a **synthetic 200-row** stand-in with the same 14 columns, so the loss values **won't match** the course's (`3,136,162 → 2,633,397`). The point is the **behavior**: the correct loop's loss **decreases steadily**; the two buggy variants stay high / frozen.

> *Verified locally:* missing `zero_grad` → loss stays high (`~5.1M → 3.9M`); `step()` before `backward()` → loss frozen (identical every epoch). Correct loop → steady decrease.


In [12]:
# Training the 14 -> 128 -> 64 -> 1 net + demonstrating the two classic bugs
# Real course: apartments_df = pd.read_csv("streeteasy.csv")  (thousands of rows)
# FAKE synthetic 200-row stand-in with the same 14 feature columns:
np.random.seed(0)
n = 200
synth = {
    'bedrooms': np.random.randint(0, 4, n), 'bathrooms': np.random.randint(1, 3, n),
    'size_sqft': np.random.randint(400, 2500, n), 'min_to_subway': np.random.randint(1, 20, n),
    'floor': np.random.randint(1, 30, n), 'building_age_yrs': np.random.randint(0, 120, n),
    'no_fee': np.random.randint(0, 2, n), 'has_roofdeck': np.random.randint(0, 2, n),
    'has_washer_dryer': np.random.randint(0, 2, n), 'has_doorman': np.random.randint(0, 2, n),
    'has_elevator': np.random.randint(0, 2, n), 'has_dishwasher': np.random.randint(0, 2, n),
    'has_patio': np.random.randint(0, 2, n), 'has_gym': np.random.randint(0, 2, n),
}
apartments_df = pd.DataFrame(synth)
apartments_df['rent'] = (2000 + 2.5*apartments_df['size_sqft']
                         + 500*apartments_df['bedrooms']
                         + np.random.randint(-300, 300, n))   # loosely rent-like target

numerical_features = ['bedrooms', 'bathrooms', 'size_sqft', 'min_to_subway', 'floor',
                      'building_age_yrs', 'no_fee', 'has_roofdeck', 'has_washer_dryer',
                      'has_doorman', 'has_elevator', 'has_dishwasher', 'has_patio', 'has_gym']
X = torch.tensor(apartments_df[numerical_features].values, dtype=torch.float)
y = torch.tensor(apartments_df['rent'].values, dtype=torch.float).view(-1, 1)


def make_model():
    torch.manual_seed(42)
    return nn.Sequential(
        nn.Linear(14, 128), nn.ReLU(),
        nn.Linear(128, 64), nn.ReLU(),
        nn.Linear(64, 1)
    )

loss = nn.MSELoss()

def train(variant, num_epochs=1000):
    model = make_model()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(num_epochs):
        predictions = model(X)
        MSE = loss(predictions, y)
        if variant == "correct":
            MSE.backward(); optimizer.step(); optimizer.zero_grad()
        elif variant == "no_zero_grad":      # CP1 bug: grads accumulate -> high loss
            MSE.backward(); optimizer.step()
        elif variant == "step_before_backward":  # CP2 bug: update before grads -> frozen
            optimizer.zero_grad(); optimizer.step(); MSE.backward()
        if (epoch + 1) % 250 == 0:
            print(f'  Epoch [{epoch+1}/{num_epochs}], MSE Loss: {MSE.item():,.1f}')

print("CP3 — correct loop (loss decreases steadily):")
train("correct")
print("\nCP1 bug — missing optimizer.zero_grad() (loss stays high):")
train("no_zero_grad", 500)
print("\nCP2 bug — step() before backward() (loss frozen):")
train("step_before_backward", 500)


CP3 — correct loop (loss decreases steadily):


  Epoch [250/1000], MSE Loss: 1,232,413.6


  Epoch [500/1000], MSE Loss: 1,119,883.2


  Epoch [750/1000], MSE Loss: 1,006,260.9


  Epoch [1000/1000], MSE Loss: 906,666.6

CP1 bug — missing optimizer.zero_grad() (loss stays high):


  Epoch [250/500], MSE Loss: 33,463,558.0


  Epoch [500/500], MSE Loss: 27,740,308.0

CP2 bug — step() before backward() (loss frozen):
  Epoch [250/500], MSE Loss: 43,913,996.0


  Epoch [500/500], MSE Loss: 43,913,996.0


## Lesson 11 — Testing & Evaluation

A falling **training** loss only proves the model is memorizing the data it *learned from*. The real question is how it does on **new** data. Two things we still need:
1. **evaluate** the model on data it has never seen
2. **save** the best model for reuse

### Train–test split
Rather than collecting new data, hold out part of the dataset: **train** on most of it, **test** on the rest. Use scikit-learn:

```python
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size=0.80,    # 80% for training
    test_size=0.20,     # 20% held out for testing
    random_state=2)     # fixed seed -> same split every run
```
- **`X`** = all input features, **`y`** = targets.
- **`random_state`** makes the split **reproducible**.
- Produces 4 tensors: train pair (`X_train`, `y_train`) and test pair (`X_test`, `y_test`).
- You **train only on the training set**, then **evaluate on the test set** — the test set stands in for "new data."

### Evaluating on the test set
```python
model.eval()
with torch.no_grad():
    predictions = model(X_test)
    test_MSE = loss(predictions, y_test)
```
- **`model.eval()`** — switches the model to **evaluation mode** (changes behavior of layers like dropout/batchnorm that act differently at train vs. test time). *(Pair it with `model.train()` to switch back before more training.)*
- **`with torch.no_grad()`** — turns off gradient tracking. We're not training, so we don't need the backward graph → **faster, less memory**.
- Otherwise it's the same as training: feedforward on the test inputs, then MSE between predictions and the true test targets.

> A **test loss much higher than the training loss** is the classic sign of **overfitting** (memorized the training data, doesn't generalize).

### Saving & loading models
So you don't retrain from scratch every time:
```python
torch.save(model, 'model.pth')          # save the whole model to a file
loaded_model = torch.load('model.pth')  # load it back
```

> **⚠️ torch ≥ 2.6 gotcha:** `torch.load` now defaults to **`weights_only=True`**, which **fails** when loading a *whole pickled model* like the line above (error: `Weights only load failed... UnpicklingError`). To load a full model you must pass **`weights_only=False`**:
> ```python
> loaded_model = torch.load('model.pth', weights_only=False)   # needed on torch 2.12
> ```
> **Best practice** (avoids the issue entirely): save just the **`state_dict`** (the weights), then load them into a freshly-built model:
> ```python
> torch.save(model.state_dict(), 'weights.pth')
> model2 = make_model()                       # rebuild the same architecture
> model2.load_state_dict(torch.load('weights.pth'))
> ```

> Maps onto the broader ML workflow (train/validation/test) — same idea as the Codecademy **Train/Validation/Test** lesson in the career-path notes.


In [13]:
# Full workflow: train-test split -> train -> evaluate on held-out test set -> save/load
from sklearn.model_selection import train_test_split
import os

# reuse the synthetic 14-feature X, y from the previous cell
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.80, test_size=0.20, random_state=2)
print(f"split -> train: {tuple(X_train.shape)},  test: {tuple(X_test.shape)}")

# train ONLY on the training set
model = make_model()
optimizer = optim.Adam(model.parameters(), lr=0.001)
for epoch in range(1000):
    MSE = loss(model(X_train), y_train)
    MSE.backward(); optimizer.step(); optimizer.zero_grad()
train_MSE = MSE.item()

# evaluate on the held-out test set
model.eval()                       # evaluation mode
with torch.no_grad():              # no gradients needed outside training
    test_MSE = loss(model(X_test), y_test)

print(f"final TRAIN MSE: {train_MSE:,.1f}")
print(f"      TEST  MSE: {test_MSE.item():,.1f}   (on data the model never trained on)")

# --- save & load ---
torch.save(model, 'model.pth')
# torch >= 2.6: whole-model load needs weights_only=False
loaded_model = torch.load('model.pth', weights_only=False)
loaded_model.eval()
with torch.no_grad():
    reloaded_test_MSE = loss(loaded_model(X_test), y_test)
print(f"reloaded model TEST MSE: {reloaded_test_MSE.item():,.1f}   (matches -> save/load worked)")

os.remove('model.pth')             # clean up the demo file


split -> train: (160, 14),  test: (40, 14)


final TRAIN MSE: 986,187.9
      TEST  MSE: 813,986.9   (on data the model never trained on)
reloaded model TEST MSE: 813,986.9   (matches -> save/load worked)


### Exercise — Train/test/save/evaluate + visualization (checkpoints 1–4)

Full end-to-end workflow on the 14-feature data:

1. **CP1 — split:** `train_test_split(X, y, train_size=0.70, test_size=0.30, random_state=2)` → **70/30** train/test.
2. **CP2 — train on the *training* set only:** the bug to fix is feeding **`X`/`y`** (the whole dataset) into the loop instead of **`X_train`/`y_train`**. Training must never see the test data, or evaluation is meaningless (data leakage).
3. **CP3 — save:** `torch.save(model, 'model.pth')`.
4. **CP4 — load a long-trained model & evaluate:** the course loads `model20k.pth` (a model trained **20,000 epochs**) and evaluates on the test set → **test MSE ≈ 1,997,977**, **RMSE ≈ 1413** ("off by ~$1,400" — decent for a tiny untuned net).

**Visualization:** scatter **predicted vs actual** rent on the test set, with a dashed **y = x** line. A perfect model puts every dot on the line; tighter clustering around it = better predictions.

> **⚠️ Faked data caveat:** we don't have the course's `streeteasy.csv` or its `model20k.pth`. The cell below **trains our own 20k-epoch model** on the synthetic 14-feature data (with ±300 noise on the target so the scatter is realistic), saves it as `model20k.pth`, then loads/evaluates/plots. The workflow + plot are faithful; the **exact numbers differ** from the course.
>
> **⚠️ torch 2.12:** `torch.load(..., weights_only=False)` is required to reload the whole model (see the Lesson 11 gotcha above).


In [14]:
# Full exercise: 70/30 split -> train on train set -> save -> load 20k model -> eval -> plot
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import os

# reuse the synthetic 14-feature X, y (target has +/-300 noise) from earlier cells

# CP1 — 70/30 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.70, test_size=0.30, random_state=2)
print(f"split -> train {tuple(X_train.shape)}, test {tuple(X_test.shape)}")

# CP2 — train ONLY on the training set (the bug = using X/y instead of X_train/y_train)
model = make_model()
loss = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 20000                      # course's pre-saved model was trained 20k epochs
for epoch in range(num_epochs):
    MSE = loss(model(X_train), y_train)
    MSE.backward(); optimizer.step(); optimizer.zero_grad()
    if (epoch + 1) % 5000 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], MSE Loss: {MSE.item():,.1f}')

# CP3 — save the trained model
torch.save(model, 'model20k.pth')

# CP4 — load it back and evaluate on the held-out test set
loaded_model = torch.load('model20k.pth', weights_only=False)   # torch>=2.6 needs this
loaded_model.eval()
with torch.no_grad():
    predictions = loaded_model(X_test)
    test_MSE = loss(predictions, y_test)

print('\nTest MSE is '      + str(test_MSE.item()))
print('Test Root MSE is ' + str(test_MSE.item()**(1/2)))   # RMSE = error in $ terms

# Visualization — predicted vs actual on the test set
plt.figure(figsize=(8, 5))
plt.scatter(y_test.numpy(), predictions.numpy(), alpha=0.5, color='blue', label='Predictions')
lo, hi = float(y_test.min()), float(y_test.max())
plt.plot([lo, hi], [lo, hi], linestyle='--', color='gray', linewidth=2, label='Perfect (y = x)')
plt.xlabel('Actual rent (y_test)'); plt.ylabel('Predicted rent')
plt.title('Synthetic StreetEasy — Predictions vs Actual (test set)')
plt.legend(); plt.tight_layout()
plt.savefig('images/l11_pred_vs_actual.png', dpi=80, bbox_inches='tight'); plt.close()
print('saved plot -> images/l11_pred_vs_actual.png')

os.remove('model20k.pth')               # clean up the demo file


split -> train (140, 14), test (60, 14)


Epoch [5000/20000], MSE Loss: 61,338.3


Epoch [10000/20000], MSE Loss: 20,219.8


Epoch [15000/20000], MSE Loss: 13,471.8


Epoch [20000/20000], MSE Loss: 10,344.2

Test MSE is 73630.328125
Test Root MSE is 271.34908904398407
saved plot -> images/l11_pred_vs_actual.png


![Predictions vs actual rent on the test set](images/l11_pred_vs_actual.png)

Dots near the dashed **y = x** line = accurate predictions. (Synthetic data, so tighter than the course's real-data scatter.)